# 🌀 **Upper-Air Analysis from Gridded Model Data**

---

### **Context**
In `02_synoptic_analysis_I.ipynb`, the upper-air overview was assembled from sparse radiosonde station plots at 500 hPa. Here we execute the operational successor to that workflow, retrieving **gridded model analysis fields** produced by operational Numerical Weather Prediction (NWP) systems through data assimilation, and performing a systematic **upper-air synoptic analysis** at 850, 700, 500, and 250 hPa. The analysis closes with an integrated diagnosis that combines the four upper-air maps with the concurrent model surface chart.

### **Learning Goals:**
- 🎯 **Goal 1 | Data Acquisition**: Retrieve NWP analysis fields for a specific domain and a target model cycle 
- 🎯 **Goal 2 | Lower-Tropospheric Thermal Structure**: Analyze the 850 hPa geopotential height, temperature, and wind fields, and identify regions of warm and cold air advection
- 🎯 **Goal 3 | Lower-to-Mid-Level Moisture**: Map the 700 hPa relative humidity field and assess the potential for cloud formation and precipitation
- 🎯 **Goal 4 | Mid-Tropospheric Dynamics**: Identify troughs and ridges at 500 hPa and compute relative vorticity advection to diagnose vertical atmospheric motions
- 🎯 **Goal 5 | Upper-Tropospheric Jet Stream**: Locate the jet stream and jet streaks from the 250 hPa wind field and identify the associated divergence and convergence regions
- 🎯 **Goal 6 | Integrated Synoptic Diagnosis**: Combine the four upper-air levels with the surface chart derived from the same GFS analysis to produce a comprehensive synoptic interpretation

**Run the notebook via the Binder platform:**

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/one-weather-lab/weather-analysis-and-forecasting/HEAD?urlpath=notebook/notebooks/03_synoptic_analysis_II.ipynb)

---

## ⚙️ Setup

First, let's import the necessary libraries and configure our environment.

In [ ]:
# Import required libraries
from __future__ import annotations

import sys
import time as _time
import warnings
from pathlib import Path

# Core data science stack
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

# Visualization
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Add local utils to path
sys.path.insert(0, str(Path('../utils').resolve()))

# Import our custom utilities
from herbie_gfs import fetch_gfs_analysis
from gfs_diagnostics import (
    compute_temperature_advection,
    compute_relative_vorticity,
    compute_relative_vorticity_advection,
    compute_wind_speed,
)
from plot_helpers import (
    plot_850hpa_gph_temperature_wind,
    plot_850hpa_temperature_advection,
    plot_700hpa_relative_humidity,
    plot_500hpa_gph,
    plot_500hpa_relative_vorticity,
    plot_500hpa_relative_vorticity_advection,
    plot_250hpa_jet,
    plot_gfs_surface_chart,
    plot_upper_air_overview,
)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up nice plotting defaults
plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print('[OK]   All libraries loaded successfully!')

---

# Section 1: Data Acquisition

## 1.1 Understanding Model Analyses

A meteorological analysis represents the **optimal three-dimensional estimate of the atmospheric state at a given time**, produced by an NWP system through data assimilation. The latter process combines observations from all components of the [Global Observing System (GOS)](https://www.google.com/url?sa=E&q=https%3A%2F%2Fwmo.int%2Factivities%2Fglobal-observing-system-gos%2Fglobal-observing-system-gos), including surface and upper-air stations, marine platforms, aircraft, satellites, and meteorological radar, with a short-range NWP forecast used as a background field. The resulting gridded fields support both synoptic diagnosis of the current atmospheric state and provision of initial conditions for subsequent NWP runs. Here we retrieve the GFS (Global Forecast System) analysis (f000) at 0.25° horizontal resolution.

> In practice, a meteorological analysis is not available at the time it represents. Observation collection, processing, and assimilation introduce a fixed delay (the **cutoff time**), so an operational analysis describes the recent rather than the instantaneous state of the atmosphere. 

## 1.2 Select Data Mode and Analysis Domain

**Instructions:**
1. **Choose your Data Mode** in the cell below (`'realtime'` for the latest observations or `'retrospective'` for historical data)
2. **Set your target date and hour** (only applicable for `'retrospective'` mode)
3. **Adjust the bounding box coordinates** to change the analysis domain
4. **Adjust contouring and smoothing parameters** to modify contour intervals and Gaussian smoothing

> In `'realtime'` mode, the notebook targets the most recently completed GFS model cycle (00, 06, 12, or 18 UTC). 

In [ ]:
# =============================================================================
# CONFIGURATION SECTION
# =============================================================================

# ---------------------------------------------------------------------------
# DATA MODE
# ---------------------------------------------------------------------------
# 'realtime'      -> target the most recent completed GFS analysis cycle.
#
# 'retrospective' -> fetch a specific historical analysis cycle defined by
#                    TARGET_DATE and TARGET_HOUR.
DATA_MODE = 'realtime'   # <-- CHANGE TO 'realtime' FOR LATEST ANALYSIS

# ---------------------------------------------------------------------------
# RETROSPECTIVE SETTINGS (used only when DATA_MODE = 'retrospective')
# ---------------------------------------------------------------------------
TARGET_DATE = '2023-09-05'   # <-- MODIFY THIS (YYYY-MM-DD)
TARGET_HOUR = 12              # <-- GFS cycle hour: 0 | 6 | 12 | 18

# ---------------------------------------------------------------------------
# TARGET DOMAIN (default: European domain, matches SA-I)
# ---------------------------------------------------------------------------
LON_MIN, LON_MAX = -25, 45
LAT_MIN, LAT_MAX =  30, 72

# ---------------------------------------------------------------------------
# CONTOUR INTERVALS
# ---------------------------------------------------------------------------
GPH_INTERVAL_850      = 2     # dam
GPH_INTERVAL_700      = 3     # dam
GPH_INTERVAL_500      = 4     # dam
GPH_INTERVAL_250      = 12    # dam
ISOTHERM_INTERVAL_850 = 1     # deg C  (0 deg C isotherm emphasized)
ISOTACH_INTERVAL_250  = 2    # m/s    (>= 50 m/s contour emphasized)
MSLP_INTERVAL         = 4     # hPa
T2M_INTERVAL          = 2     # deg C  (2 m temperature contour interval)

# ---------------------------------------------------------------------------
# SMOOTHING
# ---------------------------------------------------------------------------
SIGMA_UPPER = 3    # Gaussian smoothing sigma for upper-air fields
SIGMA_MSLP  = 3    # Gaussian smoothing sigma for MSLP

# =============================================================================
if DATA_MODE == 'retrospective':
    _valid_str = f'{TARGET_DATE} {TARGET_HOUR:02d}:00 UTC'
else:
    _valid_str = 'latest available cycle'

print(f'[DATA MODE] {DATA_MODE.upper()} | valid: {_valid_str}')
print(f'[DOMAIN   ] lon {LON_MIN} to {LON_MAX} deg  lat {LAT_MIN} to {LAT_MAX} deg')
print(f'[INTERVALS] GPH 850/700/500/250 = {GPH_INTERVAL_850}/{GPH_INTERVAL_700}/'
      f'{GPH_INTERVAL_500}/{GPH_INTERVAL_250} dam'
      f' | T850 = {ISOTHERM_INTERVAL_850} C | ISOTACH 250 = {ISOTACH_INTERVAL_250} m/s'
      f' | MSLP = {MSLP_INTERVAL} hPa | T2m = {T2M_INTERVAL} C')
print(f'[SMOOTHING] sigma_UPPER = {SIGMA_UPPER}  sigma_MSLP = {SIGMA_MSLP}')

## 1.3 Data Retrieval

The retrieval step fetches the complete set of fields required for the upper-air and surface analysis sections in a single operation. Upper-air variables cover **geopotential height, temperature, and wind at 850, 700, 500, and 250 hPa, plus relative humidity at 700 hPa**. Surface variables comprise **mean sea-level pressure and 10 m wind vectors**. The dataset is automatically subset to the domain specified in Section 1.2, and a runtime assertion confirms that the retrieved fields correspond to a GFS analysis (f000). 

In [ ]:
_start = _time.time()

ds = fetch_gfs_analysis(
    mode=DATA_MODE,
    target_date=TARGET_DATE if DATA_MODE == 'retrospective' else None,
    target_hour=TARGET_HOUR if DATA_MODE == 'retrospective' else None,
    domain=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
)

assert ds.attrs['fxx'] == 0, "Expected analysis (fxx=0); check fetch_gfs_analysis output."

print(f'[TIME  ] {(_time.time() - _start):.1f} s')
print(f'[VALID ] {ds.attrs.get("valid_time", "unknown")}')
print(f'[SOURCE] {ds.attrs.get("source", "unknown")}')
print(f'[VARS  ] {list(ds.data_vars)}')

---

# Section 2: Lower-Tropospheric Thermal Structure

## 2.1 The 850 hPa Isobaric Level

The 850 hPa isobaric level is located at an average altitude of  approximately 1,500 m, placing it just above the atmospheric boundary layer. This position minimizes the influence of surface factors (such as urban heat) and the diurnal temperature cycle, making temperature observations at this level suitable for identifying warm and cold air masses independently of observation time. Zones of strong temperature gradient, where isotherms are closely spaced, indicate **frontal boundaries**. Geopotential height is the second primary diagnostic field, contoured as isohypses. Closed isohypse patterns directly locate **lower-tropospheric cyclones (L) and anticyclones (H)**.

> Wind at this level is approximately geostrophic and flows parallel to the isohypses. Across frontal zones, wind direction shifts abruptly. 

The cell below creates an 850 hPa analysis map with geopotential height contours, air temperature shading, and wind barbs.

In [ ]:
out_path = plot_850hpa_gph_temperature_wind(
    ds,
    gph_interval=GPH_INTERVAL_850,
    isotherm_interval=ISOTHERM_INTERVAL_850,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
)
plt.show()
print(f'[OK]   Figure saved to {out_path}')

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 2.1 — Thermal Structure

**Goal:** Interpret the lower-tropospheric circulation and thermal structure from the 850 hPa analysis.

1. Identify cyclonic (L) and anticyclonic (H) centers from the isohypse pattern.
2. Locate frontal zones from the temperature gradient field and wind direction abrupt shifts.

</div>

## 2.3 Temperature Advection

Where the 850 hPa wind, blowing parallel to the isohypses, crosses the temperature field at a significant angle, temperature advection develops, approaching a maximum when the two fields are perpendicular. 

**Warm air advection (WAA)** occurs when the geostrophic wind blows from the warm side across the isotherms to the cold side. **Cold air advection (CAA)** occurs when the geostrophic wind blows from the cold side across the isotherms to the warm side. 

**WAA** forces air to ascend, which, through the mass continuity principle, is accompanied by **surface convergence**. Conversely, **CAA** induces descending vertical motion, which results in **surface divergence**. 

The cell below shades temperature advection, with WAA (positive) in red and CAA (negative) in blue. The 850 hPa geopotential height contours and wind barbs are retained.

In [ ]:
T_adv = compute_temperature_advection(
    u=ds['u_850'],
    v=ds['v_850'],
    T=ds['t_850'],
    lat=ds.latitude,
    lon=ds.longitude,
)

out_path = plot_850hpa_temperature_advection(
    ds, T_adv,
    gph_interval=GPH_INTERVAL_850,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
)
plt.show()
print(f'[OK]   Figure saved to {out_path}')

---

# Section 3: Lower-to-Mid-Level Moisture

## 3.1 The 700 hPa Isobaric Level

The 700 hPa isobaric level occupies an intermediate position between the lower and middle troposphere, at an average altitude of approximately 3,000 m. This level serves as a key reference for assessing available moisture for **cloud formation and precipitation**.

Based on relative humidity (RH), very humid conditions (RH ≥ 70 %) correspond to a high probability of cloud formation. Near-saturation conditions (RH ≥ 90 %) signal almost certain development of dense cloud cover and precipitation.

The cell below shades 700 hPa relative humidity, with geopotential height contours overlaid.

In [ ]:
out_path = plot_700hpa_relative_humidity(
    ds,
    gph_interval=GPH_INTERVAL_700,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
)
plt.show()
print(f'[OK]   Figure saved to {out_path}')

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; border-left: 4px solid #ffc107;">

## 📝 Task 3.1 — Mid-Level Moisture

**Goal:** Assess the 700 hPa moisture distribution and cross-reference it against the lower-tropospheric thermal structure diagnosed in Section 2.

1. Locate regions of very humid conditions (RH ≥ 70 %) and near-saturation (RH ≥ 90 %) on the 700 hPa map.
2. Cross-reference these regions against the WAA zones and frontal boundaries identified in Section 2, and assess whether the moisture distribution supports the expectation of cloud formation and precipitation.

</div>

---

# Section 4: Mid-Tropospheric Dynamics

## 4.1 The 500 hPa Isobaric Level

The 500 hPa level lies at approximately 5,500 m altitude. At this level, the primary diagnostic field is geopotential height. Low values of geopotential height reflect cold, compressed air in the lower tropospheric levels, whereas high values correspond to warmer, less dense air columns beneath. Accordingly, **troughs** are axes of geopotential height minima, whereas **ridges** are axes of geopotential height maxima. 

The position and orientation (tilt) of these upper-level features are key for diagnosing the **life cycle, intensification, and movement of surface pressure systems**. All other atmospheric features being equal, surface cyclones intensify more rapidly when located downstream (ahead) of a negatively tilted (SE—NW in the Northern Hemisphere) 500-hPa trough than they do when located downstream (ahead) of a positively tilted (SW—NE) 500-hPa trough. Similarly, a surface anticyclone must be located downstream (ahead) of a 500-hPa ridge axis in order to intensify. Both connections are examined through vorticity advection and associated vertical motions in Section 4.2.

The cell below contours 500 hPa geopotential height at the 4 dam interval with wind barbs overlaid, providing the basis for trough and ridge identification.

In [ ]:
out_path = plot_500hpa_gph(
    ds,
    gph_interval=GPH_INTERVAL_500,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
)
plt.show()
print(f'[OK]   Figure saved to {out_path}')

---

<div style="background-color: #d1e7dd; padding: 15px; border-radius: 8px; border-left: 4px solid #198754e0;">

## Checkpoint: Trough and ridge identification

Locate the trough and ridge axes on the map and for each one, determine whether is positively, neoutrely or negatively tilted

</div>


## 4.2 Relative Vorticity and Its Advection

Relative vorticity measures the tendency of an air parcel to rotate about a vertical axis relative to Earth's surface, with higher values located in trough regions and lower values in ridge regions. **Positive vorticity advection (PVA)** drives upper-level divergence and ascent, whereas **negative vorticity advection (NVA)** drives upper-level convergence and subsidence. 

Surface cyclones intensify when located downstream (ahead) of a 500 hPa trough axis, as established in Section 4.1, since this places PVA and upper-level divergence directly above the surface low. The more negatively tilted the 500 hPa trough, the more concentrated PVA is ahead of it, and the surface cyclone continues to intensify. Correspondingly, surface anticyclones intensify when located downstream (ahead) of a 500 hPa ridge axis, since this places the region of NVA and associated subsidence directly above it.

> **Note on cyclones' motion**: Surface cyclones are steered parallel to the 500 hPa geopotential height flow, moving toward regions where upper-level divergence drives maximum surface pressure falls. The 500 hPa trough moves in the same general direction but faster, owing to the stronger upper-tropospheric flow. Once the trough and surface low become vertically stacked, the system's westward tilt with height is eliminated and intensification ceases. 


The cell below displays a two-panel map of 500 hPa relative vorticity (left, shading) and its advection (right, PVA in red, NVA in blue), both with geopotential height contours overlaid.

In [ ]:
rvort = compute_relative_vorticity(
    u=ds['u_500'],
    v=ds['v_500'],
    lat=ds.latitude,
    lon=ds.longitude,
)

out_path_vort = plot_500hpa_relative_vorticity(
    ds, rvort,
    gph_interval=GPH_INTERVAL_500,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
    show=False,
)
print(f'[OK]   Figure saved to {out_path_vort}')

rvort_adv = compute_relative_vorticity_advection(
    u=ds['u_500'],
    v=ds['v_500'],
    rvort=rvort,
    lat=ds.latitude,
    lon=ds.longitude,
)

out_path_adv = plot_500hpa_relative_vorticity_advection(
    ds, rvort_adv,
    gph_interval=GPH_INTERVAL_500,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
    show=False,
)
print(f'[OK]   Figure saved to {out_path_adv}')

# --- Two-panel composite: relative vorticity + vorticity advection ---
fig2, axes = plt.subplots(1, 2, figsize=(26, 9))
for ax, img_path in zip(axes, [out_path_vort, out_path_adv]):
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')

plt.tight_layout()
timestamp = pd.to_datetime("now", utc=True).strftime("%Y%m%d_%H%M")
out_path_composite = Path('../outputs') / f"500hpa_vorticity_composite_{timestamp}.png"
fig2.savefig(out_path_composite, dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f'[OK]   Composite figure saved to {out_path_composite}')

---

# Section 5: Upper-Tropospheric Jet Stream

## 5.1 The 250 hPa Isobaric Level

The 250 hPa level samples the upper troposphere near the tropopause and is a key reference level for **jet stream** analysis. The jet stream is identified from the wind speed (isotach) field as a quasi-continuous band of strong winds, typically exceeding 30 m/s. Localized wind speed maxima (> 50 m/s) embedded within the jet are known as **jet streaks**. 

The classic four-quadrant jet model is applied exclusively to straight jet streaks to locate specific quadrants of **upper-level divergence** (right-entrance and left-exit) and **convergence** (left-entrance and right-exit). In contrast, a two-quadrant model is used to evaluate curved jet streaks situated around the base of a trough or the apex of a ridge. In these curved flows, vertical motions span the entire area, meaning the entire exit region of a cyclonically curved jet streak acts as a single upper-level divergent area favoring ascent and surface cyclogenesis. 

The cell below shades 250 hPa wind speed (isotachs), with the 50 m/s contour emphasized to delineate jet streaks, and overlays 250 hPa geopotential height contours and wind barbs.

In [ ]:
wspd = compute_wind_speed(
    u=ds['u_250'],
    v=ds['v_250'],
)

out_path = plot_250hpa_jet(
    ds, wspd,
    gph_interval=GPH_INTERVAL_250,
    isotach_interval=ISOTACH_INTERVAL_250,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
)
plt.show()
print(f'[OK]   Figure saved to {out_path}')

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; 
border-left: 4px solid #ffc107;">

## 📝 Task 5.1 — Jet Streak Classification and Upper-Level Forcing

**Goal:** Diagnose regions of upper-level divergence and convergence from the 250 hPa jet streak structure.

1. Identify any embedded jet streaks and classify each as straight or curved (cyclonic or anticyclonic for curved cases).
2. Apply the four-quadrant model to straight streaks and the two-quadrant model to curved streaks to locate divergence and convergence regions.

</div>

---

# Section 6: Integrated Synoptic Diagnosis

## 6.1 Surface Analysis

The cell below produces a GFS analysis surface chart comprising mean sea-level pressure isobars, 2 m air temperature contours, and 10 m wind barbs. The **surface chart** is rendered side by side with the 850 hPa temperature, geopotential height, and wind analysis from Section 2, enabling direct comparison of the surface and lower-tropospheric fields.

In [ ]:
out_path = plot_gfs_surface_chart(
    ds,
    mslp_interval=MSLP_INTERVAL,
    sigma_mslp=SIGMA_MSLP,
    t2m_interval=T2M_INTERVAL,
    output_dir='../outputs',
    show=False,
)
print(f'[OK]   Figure saved to {out_path}')

# --- Two-panel composite: surface + 850 hPa ---
path_850 = max(Path('../outputs').glob('850hpa_gph_temp_wind_*.png'), key=lambda p: p.stat().st_mtime)
fig2, axes = plt.subplots(1, 2, figsize=(26, 9))
for ax, img_path in zip(axes, [out_path, str(path_850)]):
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')

plt.tight_layout()
timestamp = pd.to_datetime("now", utc=True).strftime("%Y%m%d_%H%M")
out_path_composite = Path('../outputs') / f"surface_850hpa_composite_{timestamp}.png"
fig2.savefig(out_path_composite, dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f'[OK]   Composite figure saved to {out_path_composite}')

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; 
border-left: 4px solid #ffc107;">

## 📝 Task 6.1 — Surface Analysis and Operational Validation

**Goal:** Extend the lower-tropospheric thermal analysis from Section 2 to the surface and validate against an operational chart.

1. Examine the surface fields and use them to refine and refine the lower-tropospheric analysis from Section 2.
2. Validate your diagnosis against the UKMO operational surface 
   analysis.
   - *Real-time:* `https://www.metoffice.gov.uk/weather/maps-and-charts/surface-pressure`
   - *Retrospective:* `https://www.wetterzentrale.de/en/reanalysis.php?map=1&model=bra&var=45`

</div>

## 6.2 Four-Panel Upper-Air Overview

The cell below assembles the four upper-air diagnostics computed in Sections 2–5 into a single view at the common analysis valid time: 850 hPa thermal advection (upper left), 700 hPa relative humidity (upper right), 500 hPa vorticity advection (lower left), and 250 hPa isotachs (lower right).

These four plots collectively illustrate the three synoptic-scale mechanisms governing tropospheric **vertical motion** (lower-tropospheric thermal advection at 850 hPa, mid-tropospheric vorticity advection at 500 hPa, and upper-tropospheric jet streak forcing at 250 hPa) alongside **moisture availability** at 700 hPa, a necessary ingredient for ascent-driven cloud and precipitation development. Examined together, they support a coherent assessment of regions susceptible to organized synoptic-scale weather development. 

In [ ]:
out_path = plot_upper_air_overview(
    ds, T_adv, rvort, wspd,
    sigma=SIGMA_UPPER,
    output_dir='../outputs',
)
plt.show()
print(f'[OK]   Figure saved to {out_path}')

<div style="background-color: #fff3cd; padding: 15px; border-radius: 8px; 
border-left: 4px solid #ffc107;">

## 📝 Task 6.2 — Integrated Weather Development Assessment

**Goal:** Ground the synoptic diagnosis in observed weather.

1. Drawing on Task 6.1 and the four-panel upper-air overview, identify regions where the balance of vertical motion mechanisms favors either ascent or subsidence. For each region, characterize the expected type of weather development.
2. Validate your assessment against satellite imagery.
   - *Real-time:* EUMETSAT ProductViewer at
     `https://view.eumetsat.int/productviewer?v=default`
   - *Retrospective:* NOAA GIBBS archive at
     `https://www.ncei.noaa.gov/gibbs/year` — select the target 
     date/time and a suitable Meteosat product.

</div>

---

<div style="background-color: #f8d7da; padding: 15px; border-radius: 8px; border-left: 4px solid #dc3545;">

## Pro Task: Retrospective Case Studies

**Goal:** Apply the full upper-air and surface analysis workflow to historically significant weather events over Europe.

Return to the **Configuration** cell, switch `DATA_MODE` to `'retrospective'`, set `TARGET_DATE` and `TARGET_HOUR`, and re-run all cells to apply the analysis developed in Sections 1–6 to one of the following events:

- **Storm Ciarán** (2 Nov 2023, 00Z): An exceptionally deep and rapidly intensifying extratropical cyclone over Western Europe.
- **Heat Wave "Kleon"** (23 Jul 2023, 12Z): An extended Mediterranean heat wave affecting Greece and the central Mediterranean.
- **Storm Daniel** (5 Sep 2023, 12Z): A high-impact Mediterranean storm that produced extreme precipitation and flooding over Thessaly, central Greece.

You can also explore other dates of interest from your own research.

</div>

---

## 🏁 Summary & Key Takeaways

In this notebook, we conducted a systematic level-by-level synoptic analysis of GFS 0.25° model analysis fields over Europe, from the surface and lower troposphere to the jet stream level, and assessed the combined diagnosis against operational analyses and satellite observations.

**Key Takeaways:**

* ✅ **Lower-Tropospheric Thermal Structure:** At 850 hPa, the temperature and wind fields reveal frontal zones and diagnose lower-tropospheric temperature advection (WAA/CAA), serving as one of the three primary synoptic-scale mechanisms for vertical motion.
* ✅ **Lower-to-Mid-Level Moisture:** At 700 hPa, relative humidity reveals very humid and near-saturated regions where dynamic ascent may produce cloud formation and precipitation.
* ✅ **Mid-Tropospheric Dynamics:** At 500 hPa, trough and ridge positions and orientations govern surface pressure system development, and vorticity advection (PVA/NVA) is diagnosed as another key synoptic-scale vertical motion mechanism.
* ✅ **Upper-Tropospheric Jet Stream:** At 250 hPa, the isotach field locates the jet stream and embedded jet streaks, whose entrance and exit regions (and their specific quadrants) diagnose jet streak divergence/convergence, the final synoptic-scale vertical motion mechanism.
* ✅ **Integrated Synoptic Diagnosis:** Combining the upper-air diagnostics with the surface analysis and satellite observations supports a spatially coherent diagnosis of synoptic-scale weather development across the domain.

> **Looking Ahead:** The synoptic framework applied in this notebook identifies large-scale environments favorable to weather development, but the atmospheric conditions within those environments require further assessment. Examining vertical stability and convective potential, for instance, helps to evaluate the likelihood and severity of thunderstorms in regions where the synoptic framework diagnoses cloud formation and precipitation. These diagnostics are introduced in the next notebook through the analysis of thermodynamic profiles on Skew-T log-P diagrams.

---

## References
1. Stull, R. (2017). *Practical Meteorology: An Algebra-based Survey of Atmospheric Science*. University of British Columbia. Available at: [https://www.eoas.ubc.ca/books/Practical_Meteorology](https://www.eoas.ubc.ca/books/Practical_Meteorology)
2. Milrad, S. (2018). *Synoptic Analysis and Forecasting: An Introductory Toolkit.* Elsevier. https://doi.org/10.1016/C2015-0-05604-0 


---

**🦉 Crafted with wisdom at One Weather Lab (OWL)**<br>
Laboratory of Meteorology and Climatology, Physics Department, University of Ioannina<br>
Christos Giannaros <<chris.giannaros@uoi.gr>>